# NVIDIA Log-Returns: OLS Regression Analysis
### Econometric Portfolio Project — Marek Siuda & Gabriela Trelińska

**Period:** January 2019 – May 2025 (77 monthly observations)  
**Goal:** Explain NVIDIA (NVDA) log-returns using five macro-financial variables via OLS regression, following a Gretl-based analysis. We replicate and extend the results in Python.

---
**Variables**
| Symbol | Description | Source |
|--------|-------------|--------|
| Y | NVDA monthly log-return | Yahoo Finance |
| X1 | Log-change of FINRA margin debt | FINRA |
| X2 | VIX index level | CBOE via Yahoo Finance |
| X3 | Log-change of Fed Funds Rate | FRED |
| X4 | Log-change of Invesco QQQ ETF | Yahoo Finance |
| X5 | Log-change of USD/TWD exchange rate | Yahoo Finance |

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.stats.diagnostic as smd
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120

RESULTS_DIR = '../results/figures/'
os.makedirs(RESULTS_DIR, exist_ok=True)

## 2. Data Loading

We use the dataset assembled in Excel/Gretl (exported to CSV). All log-return variables were computed as $\Delta\ln(P_t) = \ln(P_t) - \ln(P_{t-1})$.

In [ ]:
df = pd.read_csv('../data/raw/nvda_macro_monthly.csv', parse_dates=['date'])
df = df.set_index('date').sort_index()

LABELS = {
    'y_nvda_log_ret':     'Y  — NVDA log-return',
    'x1_margin_debt_log': 'X1 — Δln(Margin Debt)',
    'x2_vix':             'X2 — VIX level',
    'x3_fed_funds_log':   'X3 — Δln(Fed Funds Rate)',
    'x4_qqq_log_ret':     'X4 — Δln(QQQ)',
    'x5_usdtwd_log':      'X5 — Δln(USD/TWD)',
}

print(f'Sample: {df.index[0].strftime("%b %Y")} – {df.index[-1].strftime("%b %Y")}  |  N = {len(df)}')
df.head()

## 3. Exploratory Data Analysis

In [ ]:
vars_of_interest = list(LABELS.keys())
df[vars_of_interest].describe().round(4)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 9))
axes = axes.ravel()

for ax, col in zip(axes, vars_of_interest):
    ax.plot(df.index, df[col], linewidth=1.2, color='steelblue')
    ax.axhline(0, color='grey', linewidth=0.6, linestyle='--')
    ax.set_title(LABELS[col], fontsize=10, fontweight='bold')
    ax.xaxis.set_major_locator(plt.MaxNLocator(6))
    ax.tick_params(axis='x', rotation=30)

fig.suptitle('Time Series of All Variables (Jan 2019 – May 2025)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR + 'time_series.png', bbox_inches='tight')
plt.show()

In [ ]:
predictors = [c for c in vars_of_interest if c != 'y_nvda_log_ret']
fig, axes = plt.subplots(1, 5, figsize=(15, 4))

for ax, col in zip(axes, predictors):
    ax.scatter(df[col], df['y_nvda_log_ret'], alpha=0.55, s=22, color='steelblue', edgecolors='none')
    m, b = np.polyfit(df[col].dropna(), df.loc[df[col].notna(), 'y_nvda_log_ret'], 1)
    x_line = np.linspace(df[col].min(), df[col].max(), 100)
    ax.plot(x_line, m * x_line + b, color='tomato', linewidth=1.4)
    ax.set_xlabel(LABELS[col].split('—')[1].strip(), fontsize=8)
    ax.set_ylabel('NVDA log-return' if col == predictors[0] else '', fontsize=8)

fig.suptitle('Scatter: NVDA Log-Return vs. Each Predictor', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR + 'scatterplots.png', bbox_inches='tight')
plt.show()

In [ ]:
corr = df[vars_of_interest].corr().round(3)
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1,
            xticklabels=[LABELS[c].split('—')[0].strip() for c in vars_of_interest],
            yticklabels=[LABELS[c].split('—')[0].strip() for c in vars_of_interest],
            ax=ax)
ax.set_title('Correlation Matrix', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR + 'correlation_matrix.png', bbox_inches='tight')
plt.show()

## 4. Baseline OLS Model (Model 1)

$$Y_t = \beta_0 + \beta_1 X1_t + \beta_2 X2_t + \beta_3 X3_t + \beta_4 X4_t + \beta_5 X5_t + \varepsilon_t$$

In [ ]:
y = df['y_nvda_log_ret']
X_base = df[['x1_margin_debt_log', 'x2_vix', 'x3_fed_funds_log',
             'x4_qqq_log_ret', 'x5_usdtwd_log']]
X_base_c = sm.add_constant(X_base)

model1 = sm.OLS(y, X_base_c).fit()
print(model1.summary())

In [ ]:
def model_summary_table(model, name='Model'):
    coef = model.params.round(5)
    pval = model.pvalues.round(4)
    sig  = pval.apply(lambda p: '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.10 else '')))
    table = pd.DataFrame({'Coef': coef, 'P-value': pval, 'Sig.': sig})
    dw = durbin_watson(model.resid)
    sep = '=' * 55
    print(f'\n{sep}')
    print(f'  {name}')
    print(f'  R^2 = {model.rsquared:.4f}   Adj. R^2 = {model.rsquared_adj:.4f}')
    print(f'  F({model.df_model:.0f}, {model.df_resid:.0f}) = {model.fvalue:.3f}   p = {model.f_pvalue:.2e}')
    print(f'  Durbin-Watson = {dw:.4f}   N = {int(model.nobs)}')
    print(f'{sep}')
    print(table.to_string())
    print('Signif. codes: *** p<0.01  ** p<0.05  * p<0.10')

model_summary_table(model1, 'Model 1 — Baseline OLS (5 regressors)')

## 5. Ramsey RESET Test

The RESET test checks whether higher-order fitted values ($\hat{y}^2$, $\hat{y}^3$) have explanatory power beyond the linear specification, indicating omitted non-linearity.

In [ ]:
reset_result = smd.linear_reset(model1, power=3, use_f=True)

print('Ramsey RESET Test (powers 2 and 3 of fitted values)')
print(f'  F-statistic : {reset_result.statistic:.4f}')
print(f'  p-value     : {reset_result.pvalue:.4f}')
print()
if reset_result.pvalue < 0.10:
    print('Reject H0 at 10% level: evidence of functional form misspecification.')
    print('We enrich the model with quadratic terms and pairwise interactions.')
else:
    print('Fail to reject H0: linear specification appears adequate.')

## 6. Extended Model with Quadratic Terms and Interactions

Following the RESET result, we augment the baseline with squared terms and all pairwise interactions, then apply sequential elimination at α = 0.10.

In [ ]:
d = df[['y_nvda_log_ret', 'x1_margin_debt_log', 'x2_vix',
        'x3_fed_funds_log', 'x4_qqq_log_ret', 'x5_usdtwd_log']].copy()

base_vars = ['x1_margin_debt_log', 'x2_vix', 'x3_fed_funds_log', 'x4_qqq_log_ret', 'x5_usdtwd_log']

for col in base_vars:
    d[col + '_sq'] = d[col] ** 2

for i, v1 in enumerate(base_vars):
    for v2 in base_vars[i+1:]:
        d[v1 + '__x__' + v2] = d[v1] * d[v2]

feature_cols = [c for c in d.columns if c != 'y_nvda_log_ret']
print(f'Extended feature set: {len(feature_cols)} regressors')

### 6.1 Sequential Backward Elimination (α = 0.10)

In [ ]:
def backward_elimination(y, features_df, alpha=0.10, verbose=True):
    remaining = list(features_df.columns)
    iteration = 0
    while True:
        X = sm.add_constant(features_df[remaining])
        model = sm.OLS(y, X).fit()
        pvals = model.pvalues.drop('const')
        max_p = pvals.max()
        if max_p > alpha:
            drop_var = pvals.idxmax()
            iteration += 1
            if verbose:
                print(f'Step {iteration}: drop [{drop_var}]  (p = {max_p:.4f})')
            remaining.remove(drop_var)
        else:
            break
    return model, remaining

final_model_ext, final_vars = backward_elimination(d['y_nvda_log_ret'], d[feature_cols])

In [ ]:
print('Final retained variables:', final_vars)
model_summary_table(final_model_ext, 'Post-Elimination Extended Model')

## 7. Final Model (Replication of Gretl Results)

The Gretl analysis converged on a parsimonious two-regressor model:

$$Y_t = \beta_0 + \beta_1 \cdot \Delta\ln(\text{QQQ})_t + \beta_2 \cdot \Delta\ln(\text{QQQ})_t \times \Delta\ln(\text{Rate})_t + \varepsilon_t$$

We reproduce it exactly here.

In [ ]:
d['qqq_x_sp'] = d['x4_qqq_log_ret'] * d['x3_fed_funds_log']

X_final = sm.add_constant(d[['x4_qqq_log_ret', 'qqq_x_sp']])
model_final = sm.OLS(d['y_nvda_log_ret'], X_final).fit()

model_summary_table(model_final, 'Final Model — const + QQQ + QQQ x DeltalnRate')
print()
print(model_final.summary())

## 8. Diagnostic Tests

We verify the Gauss-Markov assumptions for the final model.

In [ ]:
resid = model_final.resid
fitted = model_final.fittedvalues

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].scatter(fitted, resid, alpha=0.6, s=25, color='steelblue', edgecolors='none')
axes[0, 0].axhline(0, color='tomato', linewidth=1.2)
axes[0, 0].set_xlabel('Fitted values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted')

stats.probplot(resid, plot=axes[0, 1])
axes[0, 1].set_title('Normal Q-Q Plot of Residuals')
axes[0, 1].get_lines()[0].set(marker='o', markersize=4, alpha=0.6, color='steelblue')
axes[0, 1].get_lines()[1].set(color='tomato')

axes[1, 0].scatter(fitted, np.sqrt(np.abs(resid)), alpha=0.6, s=25, color='steelblue', edgecolors='none')
axes[1, 0].set_xlabel('Fitted values')
axes[1, 0].set_ylabel('sqrt(|Residuals|)')
axes[1, 0].set_title('Scale-Location')

axes[1, 1].plot(d.index, resid, linewidth=1.0, color='steelblue')
axes[1, 1].axhline(0, color='tomato', linewidth=1.0, linestyle='--')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('Residuals')
axes[1, 1].set_title('Residuals over Time')
axes[1, 1].tick_params(axis='x', rotation=30)

fig.suptitle('Diagnostic Plots — Final Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR + 'diagnostics.png', bbox_inches='tight')
plt.show()

In [ ]:
bp_lm, bp_p, bp_fval, bp_fp = smd.het_breuschpagan(resid, X_final)
jb_stat, jb_p, jb_skew, jb_kurt = jarque_bera(resid)
dw_stat = durbin_watson(resid)
lb_result = smd.acorr_ljungbox(resid, lags=[12], return_df=True)
lb_p_val = lb_result['lb_pvalue'].iloc[0]
lb_q_val = lb_result['lb_stat'].iloc[0]

bp_verdict  = 'Reject H0: heteroskedastic'     if bp_p < 0.05 else 'No evidence of heteroskedasticity'
jb_verdict  = 'Reject H0: non-normal residuals' if jb_p < 0.05 else 'Residuals appear normally distributed'
dw_verdict  = ('Potential positive autocorrelation' if dw_stat < 1.5
               else 'Potential negative autocorrelation' if dw_stat > 2.5
               else 'No strong evidence of autocorrelation')
lb_verdict  = 'Reject H0: autocorrelation present' if lb_p_val < 0.05 else 'No significant autocorrelation'

sep = '=' * 55
print('Diagnostic Tests — Final Model')
print(sep)
print('Breusch-Pagan (heteroskedasticity):')
print(f'  LM stat = {bp_lm:.4f}   p = {bp_p:.4f}   -> {bp_verdict}')
print()
print('Jarque-Bera (normality of residuals):')
print(f'  JB stat = {jb_stat:.4f}   p = {jb_p:.4f}   -> {jb_verdict}')
print(f'  Skewness = {jb_skew:.4f}   Excess kurtosis = {jb_kurt:.4f}')
print()
print('Durbin-Watson (autocorrelation):')
print(f'  DW = {dw_stat:.4f}   -> {dw_verdict}')
print()
print('Ljung-Box Q-test (lag 12):')
print(f'  Q = {lb_q_val:.4f}   p = {lb_p_val:.4f}   -> {lb_verdict}')

In [ ]:
X_vif = sm.add_constant(d[['x4_qqq_log_ret', 'qqq_x_sp']])
vif_data = pd.DataFrame({
    'Variable': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
print('Variance Inflation Factors (VIF):')
print(vif_data.to_string(index=False))
print('(VIF > 10 indicates severe multicollinearity)')

## 9. Actual vs. Predicted

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(d.index, d['y_nvda_log_ret'], label='Actual', linewidth=1.3, color='steelblue')
axes[0].plot(d.index, fitted, label='Fitted', linewidth=1.3, color='tomato', linestyle='--')
axes[0].axhline(0, color='grey', linewidth=0.5)
axes[0].set_title('NVDA Log-Return: Actual vs. Fitted', fontweight='bold')
axes[0].set_ylabel('Log-return')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=30)

axes[1].scatter(fitted, d['y_nvda_log_ret'], alpha=0.6, s=30, color='steelblue', edgecolors='none')
lims = [min(fitted.min(), d['y_nvda_log_ret'].min()) - 0.02,
        max(fitted.max(), d['y_nvda_log_ret'].max()) + 0.02]
axes[1].plot(lims, lims, 'r--', linewidth=1.2, label='45-degree line')
axes[1].set_xlabel('Fitted values')
axes[1].set_ylabel('Actual values')
axes[1].set_title(f'Actual vs. Fitted  (R2 = {model_final.rsquared:.3f})', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR + 'actual_vs_fitted.png', bbox_inches='tight')
plt.show()

## 10. Results Summary

| | Baseline OLS | Final Model |
|---|---|---|
| **Regressors** | 5 (X1–X5) | const + X4 + X4×X3 |
| **R²** | ~0.62 | ~0.64 |
| **F-statistic** | significant | p < 10⁻¹⁶ |
| **N** | 77 | 77 |

### Interpretation

**QQQ log-return (X4)** is the dominant driver of NVDA monthly log-returns. The Nasdaq-100 ETF captures broad tech-sector momentum; NVIDIA, as a high-beta constituent, amplifies these moves.

**QQQ × ΔlnRate interaction**: The interaction term shows that the sensitivity of NVDA returns to QQQ is itself modulated by the direction of interest rate changes. Rising rates dampen the pass-through from QQQ to NVDA (consistent with higher discount rates compressing growth-stock valuations), while falling rates amplify it.

**RESET test** rejected the purely linear baseline at α = 0.10, motivating the enriched specification. Sequential backward elimination identified the interaction as the only non-linear term with robust marginal explanatory power.

**Residual diagnostics** are reported above. Any heteroskedasticity would motivate White/HAC robust standard errors (a natural extension).